# Myotome 3D Gene-Expression Interpolation

Compare VTK, kernel, Gaussian-process, and neural interpolation on a reconstructed right-myotome voxel model.

This curated notebook targets the current Dynamo-free Spateo API. Edit the path/configuration cells for a new system before execution.


In [ ]:
import numpy as np

import pyvista as pv

pv.global_theme.transparent_background = True
import spateo as st


## Load and validate data


In [ ]:
Myotome_right = st.read_h5ad("/DATA/User/gaomohan/DATA/CS13_Project/cs13/Myotome_v2_right.h5ad")


In [ ]:
if "counts_X" not in Myotome_right.layers:
    Myotome_right.layers["counts_X"] = Myotome_right.X.copy()
st.pp.normalize_total(
    Myotome_right,
    layer="counts_X",
    out_layer="norm_X",
    target_sum=None,
    size_factor_key="Size_Factor",
    inplace=True,
)
st.pp.log1p_layer(Myotome_right, layer="norm_X", out_layer="log1p_X", set_X=True, inplace=True)


In [ ]:
cpo = [
    (7235.672822135923, -9333.743725966886, 29537.530999873827),
    (5265.3827, 317.52565000000004, 1200.0),
    (-0.9910735612049205, -0.13108990476291582, 0.024261763123203748),
]


## Construct the point-cloud model


In [ ]:
Myotome_right_pc, plot_cmap = st.tdr.construct_pc(
    adata=Myotome_right.copy(),
    spatial_key="spatial",
    groupby="celltype",
    key_added="tissue",
    colormap="#00BFC4",
)


## Reconstruct the surface mesh


In [ ]:
Myotome_right_mesh, _, _ = st.tdr.construct_surface(
    pc=Myotome_right_pc,
    key_added="tissue",
    alpha=0.6,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 1.32},
    smooth=5000,
    scale_factor=1.02,
)


## Reconstruct the voxel model


In [ ]:
Myotome_right_voxel, _ = st.tdr.voxelize_mesh(
    mesh=Myotome_right_mesh, voxel_pc=Myotome_right_pc, key_added="tissue", smooth=120
)
Myotome_right_voxel


In [ ]:
genes = ["HOXD3"]
# DDIT4 HSP90AA1 ["VIM","MEGF10"]["PEG10","CADM1"]"VIM","TTN","IGF2","PAX3","HOXD3"

# Add gene expression matrix to the point cloud model
pc_index = Myotome_right_pc.point_data["obs_index"].tolist()
for gene_name in genes:
    exp = Myotome_right[pc_index, gene_name].X.toarray().ravel()
    st.tdr.add_model_labels(
        model=Myotome_right_pc, labels=exp, key_added=gene_name, where="point_data", inplace=True
    )


In [ ]:
st.pl.three_d_multi_plot(
    model=Myotome_right_pc,
    key=genes,
    colormap="hot_r",
    opacity=0.5,
    model_style="points",
    jupyter="static",
    window_size=(800, 800),
    cpo=[cpo],
    # filename = f"./e9.5_heart_raw_gene_exp.pdf"
)


In [ ]:
from scipy import sparse

Myotome_right_dense = Myotome_right.copy()
if sparse.issparse(Myotome_right_dense.X):
    Myotome_right_dense.X = Myotome_right_dense.X.toarray()


# VTK interpolation


## Interpolate gene expression in 3D


In [ ]:
interpolated_vtk_adata = st.tdr.vtk_interpolation(
    source_adata=Myotome_right_dense.copy(),
    spatial_key="spatial",
    keys=genes,
    target_points=np.asarray(Myotome_right_voxel.points),
    n_points=5,
)
interpolated_vtk_adata


In [ ]:
interpolated_vtk_pc, _ = st.tdr.construct_pc(
    adata=interpolated_vtk_adata.copy(), spatial_key="spatial", groupby=genes[0]
)
_pc_index = interpolated_vtk_pc.point_data["obs_index"].tolist()
for gene_name in genes[1:]:
    _exp = interpolated_vtk_adata[_pc_index, gene_name].X.flatten()
    st.tdr.add_model_labels(
        model=interpolated_vtk_pc,
        labels=_exp,
        key_added=gene_name,
        where="point_data",
        inplace=True,
    )

st.pl.three_d_multi_plot(
    model=interpolated_vtk_pc,
    key=genes,
    colormap="hot_r",
    opacity=1.0,
    model_style="points",
    jupyter="static",
    cpo=[None],
)


# SparseVFC kernel interpolation


In [ ]:
interpolated_svfc_adata = st.tdr.kernel_interpolation(
    source_adata=Myotome_right_dense.copy(),
    spatial_key="spatial",
    keys=genes,
    target_points=np.asarray(Myotome_right_voxel.points),
)
interpolated_svfc_adata


In [ ]:
interpolated_svfc_pc, _ = st.tdr.construct_pc(
    adata=interpolated_svfc_adata.copy(), spatial_key="spatial", groupby=genes[0]
)
_pc_index = interpolated_svfc_pc.point_data["obs_index"].tolist()

for gene_name in genes[1:]:
    _exp = interpolated_svfc_adata[_pc_index, gene_name].X.flatten()
    st.tdr.add_model_labels(
        model=interpolated_svfc_pc,
        labels=_exp,
        key_added=gene_name,
        where="point_data",
        inplace=True,
    )

st.pl.three_d_multi_plot(
    model=interpolated_svfc_pc,
    key=genes,
    colormap="hot_r",
    opacity=0.5,
    model_style="points",
    jupyter="static",
    cpo=[cpo],
    window_size=(800, 800),
    show_legend=False,
)


# Gaussian process interpolation


In [ ]:
interpolated_gp_adata = st.tdr.gp_interpolation(
    source_adata=Myotome_right_dense,
    spatial_key="spatial",
    keys=genes,
    target_points=np.asarray(Myotome_right_voxel.points),
    device="cpu",
)
interpolated_gp_adata


In [ ]:
interpolated_gp_pc, _ = st.tdr.construct_pc(
    adata=interpolated_gp_adata.copy(), spatial_key="spatial", groupby=genes[0]
)
_pc_index = interpolated_gp_pc.point_data["obs_index"].tolist()

for gene_name in genes[1:]:
    _exp = interpolated_gp_adata[_pc_index, gene_name].X.flatten()
    st.tdr.add_model_labels(
        model=interpolated_gp_pc, labels=_exp, key_added=gene_name, where="point_data", inplace=True
    )

st.pl.three_d_multi_plot(
    model=interpolated_gp_pc,
    key=genes,
    colormap="hot_r",
    opacity=0.5,
    model_style="points",
    jupyter="static",
    cpo=[cpo],
    window_size=(800, 800),
)


### Neural interpolation


In [ ]:
interpolated_deep_adata = st.tdr.deep_intepretation(
    source_adata=Myotome_right_dense.copy(),
    spatial_key="spatial",
    keys=genes,
    target_points=np.asarray(Myotome_right_voxel.points),
)
interpolated_deep_adata


In [ ]:
interpolated_deep_pc, _ = st.tdr.construct_pc(
    adata=interpolated_deep_adata.copy(), spatial_key="spatial", groupby=genes[0]
)
_pc_index = interpolated_deep_pc.point_data["obs_index"].tolist()
for gene_name in genes[1:]:
    _exp = interpolated_deep_adata[_pc_index, gene_name].X.flatten()
    st.tdr.add_model_labels(
        model=interpolated_deep_pc,
        labels=_exp,
        key_added=gene_name,
        where="point_data",
        inplace=True,
    )

st.pl.three_d_multi_plot(
    model=interpolated_deep_pc,
    key=genes,
    colormap="hot_r",
    opacity=0.5,
    model_style="points",
    jupyter="static",
    cpo=[cpo],
)


# Slice a certain interpolation result


In [ ]:
for gene_name in genes:
    Myotome_right_voxel.point_data[gene_name] = np.asarray(interpolated_gp_adata[:, gene_name].X)
voxel_slices_x = st.tdr.three_d_slice(
    model=Myotome_right_voxel, method="axis", n_slices=20, axis="x"
)


In [ ]:
GLOBAL_CLIM = {}

for gene in genes:
    values = np.asarray(Myotome_right_voxel.point_data[gene]).reshape(-1)

    GLOBAL_CLIM[gene] = (
        float(np.nanmin(values)),
        float(np.nanmax(values)),
    )


In [ ]:
gene = genes[0]

p = pv.Plotter()

for sl in voxel_slices_x:
    if sl is None or sl.n_points == 0:
        continue

    p.add_mesh(
        sl,
        scalars=gene,
        cmap="hot_r",
        clim=GLOBAL_CLIM[gene],  # Use one fixed range for every slice.
        ambient=0.5,
        show_scalar_bar=False,
    )

p.add_scalar_bar(title=gene)

p.show(jupyter_backend="static")


In [ ]:
for gene_name in genes:
    Myotome_right_voxel.point_data[gene_name] = np.asarray(interpolated_gp_adata[:, gene_name].X)
voxel_slices_x = st.tdr.three_d_slice(
    model=Myotome_right_voxel, method="axis", n_slices=20, axis="y"
)


In [ ]:
gene = genes[0]

global_exp = np.asarray(Myotome_right_voxel.point_data[gene]).reshape(-1)

print("global:", np.nanmin(global_exp), np.nanmax(global_exp))

for i, sl in enumerate(voxel_slices_x):

    if sl is None or sl.n_points == 0:
        continue

    exp = np.asarray(sl.point_data[gene]).reshape(-1)

    print(f"slice {i:02d}: " f"min={np.nanmin(exp):.6f}, " f"max={np.nanmax(exp):.6f}")


In [ ]:
adata_xyz = np.asarray(interpolated_vtk_adata.obsm["spatial"])

voxel_xyz = np.asarray(Myotome_right_voxel.points)

print("shape:")
print(adata_xyz.shape)
print(voxel_xyz.shape)

print("Strictly identical order:", np.allclose(adata_xyz, voxel_xyz, atol=1e-8, rtol=0))

errors = np.linalg.norm(adata_xyz - voxel_xyz, axis=1)

print("max coordinate error:", errors.max())

print("mean coordinate error:", errors.mean())


In [ ]:
gene = genes[0]

for i in [46, 47, 48, 49]:
    sl = voxel_slices_x[i]

    vals = np.asarray(sl.point_data[gene]).reshape(-1)
    imax = np.nanargmax(vals)

    p = np.asarray(sl.points[imax])

    print(f"slice {i}: " f"value={vals[imax]:.6f}, " f"coord={p}")


In [ ]:
gene = genes[0]

xyz = np.asarray(interpolated_vtk_adata.obsm["spatial"])

exp = np.asarray(interpolated_vtk_adata[:, gene].X).reshape(-1)

for i in [0, 10, 11, 12, 13, 46, 47, 48, 49]:

    x0 = np.mean(voxel_slices_x[i].points[:, 0])

    # Select the voxel layer nearest to the requested slice.
    # The slice spacing is approximately 148, so begin with a +/-75 window.
    mask = np.abs(xyz[:, 0] - x0) < 75

    vals = exp[mask]

    print(
        f"slice {i:02d}, "
        f"x={x0:.2f}, "
        f"n={mask.sum()}, "
        f"min={np.nanmin(vals):.6f}, "
        f"max={np.nanmax(vals):.6f}, "
        f"mean={np.nanmean(vals):.6f}"
    )


In [ ]:
xyz = np.asarray(interpolated_vtk_adata.obsm["spatial"])

exp = np.asarray(interpolated_vtk_adata[:, gene].X).reshape(-1)

slice_x = np.array([np.mean(sl.points[:, 0]) for sl in voxel_slices_x])

half_step = np.median(np.diff(slice_x)) / 2

for i, x0 in enumerate(slice_x):

    mask = (xyz[:, 0] >= x0 - half_step) & (xyz[:, 0] < x0 + half_step)

    if not np.any(mask):
        continue

    vals = exp[mask]

    print(
        f"{i:02d}: "
        f"x={x0:8.1f} | "
        f"point max={np.nanmax(vals):.4f} | "
        f"slice max="
        f"{np.nanmax(voxel_slices_x[i].point_data[gene]):.4f}"
    )


In [ ]:
gene = genes[0]

a = np.asarray(interpolated_vtk_adata[:, gene].X).reshape(-1)

b = np.asarray(Myotome_right_voxel.point_data[gene]).reshape(-1)

print("gene:", gene)

print("interpolated_vtk_adata:", a.min(), a.max(), a.mean())

print("voxel point_data:", b.min(), b.max(), b.mean())

print("allclose:", np.allclose(a, b))

print("max abs difference:", np.max(np.abs(a - b)))

print("first 20:")
for i in range(20):
    print(i, a[i], b[i])
